# YouTube Learning Assistant

**RAG application for querying YouTube video transcripts with timestamp navigation**

Ask questions about video content and jump directly to relevant moments.

Week 5 Contribution - dc_dalin

## Setup

## Installation

Run the cell below to install dependencies, then restart the kernel before running other cells.

In [ ]:
%pip install -q openai python-dotenv gradio chromadb youtube-transcript-api

In [10]:
import os
import re
from typing import List, Dict, Tuple
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from chromadb import PersistentClient
from youtube_transcript_api import YouTubeTranscriptApi
import gradio as gr

# Load .env from project root or current directory
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if OPENAI_API_KEY:
    print(f"✅ API Key loaded: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️ Set OPENAI_API_KEY in environment or .env file")

openai = OpenAI()
MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-large"
DB_PATH = "youtube_db"
COLLECTION = "transcripts"

✅ API Key loaded: sk-proj-...


## Part 1: Index YouTube Videos

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract video ID from YouTube URL"""
    patterns = [r'(?:v=|\/)([0-9A-Za-z_-]{11})', r'(?:embed\/)([0-9A-Za-z_-]{11})', r'^([0-9A-Za-z_-]{11})$']
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def fetch_and_chunk_transcript(video_id: str, chunk_duration: int = 120) -> Tuple[List[Dict], str]:
    """Fetch transcript and split into ~2 minute chunks with overlap
    
    Returns:
        Tuple of (chunks, error_message). If successful, error_message is empty.
    """
    try:
        print(f"🔍 Attempting to fetch transcript for video: {video_id}")
        # Create API instance and fetch transcript
        api = YouTubeTranscriptApi()
        transcript = api.fetch(video_id)
        print(f"✅ Transcript fetched: {len(transcript)} entries")
    except Exception as e:
        error_msg = str(e)
        print(f"❌ Transcript fetch error: {error_msg}")
        
        # Provide more helpful error messages
        if "Could not retrieve a transcript" in error_msg or "No transcripts were found" in error_msg:
            return [], "No captions/subtitles available for this video"
        elif "Video unavailable" in error_msg:
            return [], "Video is unavailable or private"
        elif "Subtitles are disabled" in error_msg or "disabled" in error_msg.lower():
            return [], "Subtitles are disabled for this video"
        else:
            return [], f"Error fetching transcript: {error_msg}"
    
    if not transcript:
        return [], "Transcript is empty"
    
    chunks = []
    current_text = []
    start_time = 0
    duration = 0
    
    # Note: transcript items have .text, .start, .duration attributes
    for entry in transcript:
        current_text.append(entry.text)
        duration += entry.duration
        
        if duration >= chunk_duration:
            chunks.append({
                'text': ' '.join(current_text),
                'video_id': video_id,
                'start_time': start_time,
                'url': f"https://youtube.com/watch?v={video_id}&t={int(start_time)}s"
            })
            
            overlap = current_text[-2:] if len(current_text) > 2 else []
            current_text = overlap
            start_time += duration - (len(overlap) * 5)
            duration = len(overlap) * 5
    
    if current_text:
        chunks.append({
            'text': ' '.join(current_text),
            'video_id': video_id,
            'start_time': start_time,
            'url': f"https://youtube.com/watch?v={video_id}&t={int(start_time)}s"
        })
    
    print(f"📦 Created {len(chunks)} chunks from transcript")
    return chunks, ""


def index_video(video_url: str) -> Tuple[int, str]:
    """Index a YouTube video into ChromaDB
    
    Returns:
        Tuple of (num_chunks, error_message). If successful, error_message is empty.
    """
    video_id = extract_video_id(video_url)
    if not video_id:
        print("❌ Invalid URL format")
        return 0, "Invalid YouTube URL format"
    
    print(f"📥 Processing video: {video_id}")
    chunks, error = fetch_and_chunk_transcript(video_id)
    
    if error:
        return 0, error
    
    if not chunks:
        return 0, "No transcript chunks created"
    
    try:
        print(f"🔧 Embedding {len(chunks)} chunks...")
        
        chroma = PersistentClient(path=DB_PATH)
        collection = chroma.get_or_create_collection(COLLECTION)
        
        texts = [c['text'] for c in chunks]
        embeddings = [e.embedding for e in openai.embeddings.create(
            model=EMBEDDING_MODEL, input=texts
        ).data]
        
        existing = collection.count()
        ids = [f"chunk_{existing + i}" for i in range(len(chunks))]
        metadatas = [{k: v for k, v in c.items() if k != 'text'} for c in chunks]
        
        collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        
        total = collection.count()
        print(f"✅ Indexed {len(chunks)} chunks (Total in DB: {total})")
        return len(chunks), ""
    except Exception as e:
        error_msg = f"Database/Embedding error: {str(e)}"
        print(f"❌ {error_msg}")
        return 0, error_msg

## Part 2: Query Videos

In [13]:
def search_videos(query: str, k: int = 5) -> List[Dict]:
    """Search for relevant video segments"""
    query_embedding = openai.embeddings.create(
        model=EMBEDDING_MODEL, input=[query]
    ).data[0].embedding
    
    chroma = PersistentClient(path=DB_PATH)
    try:
        collection = chroma.get_collection(COLLECTION)
    except:
        return []
    
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    
    if not results['documents'] or not results['documents'][0]:
        return []
    
    sources = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        sources.append({
            'text': doc,
            'video_id': meta['video_id'],
            'start_time': meta['start_time'],
            'url': meta['url']
        })
    return sources


def answer_question(question: str, history: List = None) -> Tuple[str, List[Dict]]:
    """Answer question using RAG"""
    sources = search_videos(question, k=8)
    
    if not sources:
        return "❌ No videos indexed. Please index videos first.", []
    
    context = "\n\n".join([
        f"[Segment {i+1} from video {s['video_id']} at {int(s['start_time']//60)}:{int(s['start_time']%60):02d}]\n{s['text']}"
        for i, s in enumerate(sources[:5])
    ])
    
    system_prompt = f"""You are a helpful assistant that answers questions based on YouTube video transcripts.

Context from videos:
{context}

Answer the user's question based on the transcripts above. Be concise and accurate.
If the answer isn't in the transcripts, say so."""
    
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": question})
    
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.7
    )
    
    return response.choices[0].message.content, sources[:3]

## Part 3: Test It Out

In [ ]:
# Example: Index a video
# test_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
# num_chunks, error = index_video(test_url)
# if error:
#     print(f"Error: {error}")
# else:
#     print(f"Success! Indexed {num_chunks} chunks")

In [6]:
# Example: Ask a question
# answer, sources = answer_question("What is the main topic of this video?")
# print(f"Answer: {answer}\n")
# for i, source in enumerate(sources, 1):
#     print(f"{i}. {source['url']}")

## Part 4: Gradio Interface

In [ ]:
def format_sources(sources: List[Dict]) -> str:
    """Format sources as HTML"""
    if not sources:
        return "<p>No sources</p>"
    
    html = "<h3>📺 Video Segments</h3>"
    for i, s in enumerate(sources, 1):
        mins, secs = int(s['start_time']//60), int(s['start_time']%60)
        html += f"""
        <div style='margin:12px 0; padding:12px; background:#f5f5f5; border-radius:8px'>
            <p><b>Segment {i}</b> - Video {s['video_id']} at {mins}:{secs:02d}</p>
            <p style='color:#555'>{s['text'][:200]}...</p>
            <a href='{s['url']}' target='_blank' style='color:#1a73e8'>▶️ Watch at {mins}:{secs:02d}</a>
        </div>
        """
    return html


def index_ui(url):
    """UI wrapper for indexing videos with improved error messages"""
    if not url.strip():
        return "❌ Please enter a YouTube URL"
    
    try:
        n, error = index_video(url)
        if error:
            return f"❌ {error}"
        elif n > 0:
            return f"✅ Successfully indexed {n} chunks!"
        else:
            return "❌ No chunks were created (unknown error)"
    except Exception as e:
        return f"❌ Unexpected error: {str(e)}"


def chat_ui(message, history):
    """Chat UI with proper message format handling"""
    if not message.strip():
        return history, ""
    
    # history is already in messages format with type="messages"
    chat_history = history if history else []
    
    answer, sources = answer_question(message, chat_history)
    
    # Append new messages
    new_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    
    return new_history, format_sources(sources)


with gr.Blocks(title="YouTube Learning Assistant") as app:
    gr.Markdown("""
    # 🎓 YouTube Learning Assistant
    
    Index YouTube videos and ask questions about their content.
    Get answers with direct links to specific video moments.
    """)
    
    with gr.Tabs():
        with gr.Tab("📚 Index Videos"):
            gr.Markdown("### Add a YouTube Video")
            gr.Markdown("Paste a YouTube URL. Video must have captions/subtitles available.")
            
            url_input = gr.Textbox(
                label="YouTube URL",
                placeholder="https://www.youtube.com/watch?v=..."
            )
            index_btn = gr.Button("Index Video", variant="primary")
            result_output = gr.Textbox(label="Result", lines=3)
            
            gr.Markdown("""
            **Tips:**
            - Make sure the video has captions enabled (look for CC button)
            - Most educational/tutorial videos have auto-generated captions
            - Try this example: `https://www.youtube.com/watch?v=dQw4w9WgXcQ`
            """)
            
            index_btn.click(index_ui, inputs=[url_input], outputs=[result_output])
        
        with gr.Tab("💬 Ask Questions"):
            gr.Markdown("### Search Your Videos")
            
            with gr.Row():
                with gr.Column(scale=2):
                    chatbot = gr.Chatbot(label="Chat", height=500, type="messages")
                    question_input = gr.Textbox(
                        label="Your Question",
                        placeholder="What did the video explain about...?"
                    )
                    with gr.Row():
                        send_btn = gr.Button("Send", variant="primary")
                        clear_btn = gr.Button("Clear")
                
                with gr.Column(scale=1):
                    sources_html = gr.HTML("<p style='color:#999'>Sources will appear here</p>")
            
            question_input.submit(
                chat_ui, [question_input, chatbot], [chatbot, sources_html]
            ).then(lambda: "", outputs=[question_input])
            
            send_btn.click(
                chat_ui, [question_input, chatbot], [chatbot, sources_html]
            ).then(lambda: "", outputs=[question_input])
            
            clear_btn.click(
                lambda: ([], ""),
                outputs=[chatbot, sources_html]
            )
        
        with gr.Tab("ℹ️ Help"):
            gr.Markdown("""
            ## How to Use
            
            1. **Index Videos**: Paste YouTube URLs in the "Index Videos" tab
            2. **Ask Questions**: Go to "Ask Questions" tab and chat about video content
            3. **Click Timestamps**: Direct links to relevant video moments
            
            ## Features
            
            - Semantic search across video transcripts
            - Timestamp-aware chunking with overlap
            - Direct links to video moments
            - Multi-video knowledge base
            - Conversation memory
            
            ## Requirements
            
            - Videos must have captions/transcripts available
            - OpenAI API key in `.env` file
            
            ## Common Issues
            
            - **No captions available**: The video must have either manual or auto-generated captions
            - **Video unavailable**: The video may be private, age-restricted, or deleted
            - Check the terminal output for detailed error messages
            
            ## Example Questions
            
            - "What were the main points discussed?"
            - "How does [concept] work?"
            - "What examples were given?"
            - "Summarize the key takeaways"
            """)

app.launch(inbrowser=True)

## Summary

This RAG application:
- ✅ Fetches YouTube transcripts automatically
- ✅ Chunks transcripts with timestamps and overlap
- ✅ Stores embeddings in ChromaDB
- ✅ Semantic search across videos
- ✅ Generates answers with source citations
- ✅ Direct links to specific video moments
- ✅ Clean Gradio UI

**Week 5 - dc_dalin**